In [ ]:
# 0) setup: session clock, embedded code, data locations
import base64, glob, hashlib, io, json, os, shlex, shutil, subprocess, sys, tarfile, threading, time
T0 = time.time()
LIMIT = T0 + 11.75 * 3600            # Kaggle stops the session at 12 h
CODE, OUT = '/kaggle/working/code', '/kaggle/working/out'
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == 'e57f3fdd56b88802c509f3ef761204c0a408cc1d0c27a28bc58c2f881a8c399c'
tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz').extractall(CODE)
os.makedirs(OUT, exist_ok=True)
print('code sha256 e57f3fdd56b88802c509f3ef761204c0a408cc1d0c27a28bc58c2f881a8c399c from git 233c70d+local changes', flush=True)
subprocess.run('nvidia-smi --query-gpu=index,name,memory.total --format=csv; free -g | head -2; nproc', shell=True)
QUEUES = [[{'name': 'bench-cave-x4-puformer', 'dataset': 'cave', 'scale': 4, 'model': 'puformer', 'args': '--dataset cave --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1', 'hours': 4.0, 'robust': True, 'rargs': '--dataset cave --scale 4 --model puformer --pad reflect'}, {'name': 'bench-cave-x4-dct', 'dataset': 'cave', 'scale': 4, 'model': 'dct', 'args': '--dataset cave --scale 4 --model dct --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset cave --scale 4 --model dct'}, {'name': 'bench-harvard-x4-puformer', 'dataset': 'harvard', 'scale': 4, 'model': 'puformer', 'args': '--dataset harvard --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1', 'hours': 4.0, 'robust': True, 'rargs': '--dataset harvard --scale 4 --model puformer --pad reflect'}, {'name': 'bench-harvard-x4-dct', 'dataset': 'harvard', 'scale': 4, 'model': 'dct', 'args': '--dataset harvard --scale 4 --model dct --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset harvard --scale 4 --model dct'}, {'name': 'bench-pavia-x8-puformer', 'dataset': 'pavia', 'scale': 8, 'model': 'puformer', 'args': '--dataset pavia --scale 8 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1', 'hours': 4.0, 'robust': True, 'rargs': '--dataset pavia --scale 8 --model puformer --pad reflect'}, {'name': 'bench-pavia-x8-dct', 'dataset': 'pavia', 'scale': 8, 'model': 'dct', 'args': '--dataset pavia --scale 8 --model dct --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 8 --model dct'}, {'name': 'bench-pavia-x8-ssrnet', 'dataset': 'pavia', 'scale': 8, 'model': 'ssrnet', 'args': '--dataset pavia --scale 8 --model ssrnet --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 1.5, 'robust': False, 'rargs': '--dataset pavia --scale 8 --model ssrnet'}, {'name': 'bench-cave-x8-dct', 'dataset': 'cave', 'scale': 8, 'model': 'dct', 'args': '--dataset cave --scale 8 --model dct --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset cave --scale 8 --model dct'}, {'name': 'bench-pavia-x16-puformer', 'dataset': 'pavia', 'scale': 16, 'model': 'puformer', 'args': '--dataset pavia --scale 16 --model puformer --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1', 'hours': 4.0, 'robust': True, 'rargs': '--dataset pavia --scale 16 --model puformer --pad reflect'}, {'name': 'bench-pavia-x16-dct', 'dataset': 'pavia', 'scale': 16, 'model': 'dct', 'args': '--dataset pavia --scale 16 --model dct --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 16 --model dct'}, {'name': 'bench-chikusei-x16-puformer', 'dataset': 'chikusei', 'scale': 16, 'model': 'puformer', 'args': '--dataset chikusei --scale 16 --model puformer --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1', 'hours': 4.0, 'robust': True, 'rargs': '--dataset chikusei --scale 16 --model puformer --pad reflect'}, {'name': 'bench-chikusei-x16-dct', 'dataset': 'chikusei', 'scale': 16, 'model': 'dct', 'args': '--dataset chikusei --scale 16 --model dct --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset chikusei --scale 16 --model dct'}, {'name': 'abl-pavia-x4-nophys', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --physics 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --physics 0'}, {'name': 'abl-pavia-x4-zeros', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --pad zeros', 'hours': 4.0, 'robust': True, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --pad zeros'}, {'name': 'abl-pavia-x4-nomem', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --memory 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --memory 0'}, {'name': 'abl-pavia-x4-k2', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --stages 2', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --stages 2'}, {'name': 'abl-pavia-x4-conv', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --mixer conv', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --mixer conv'}, {'name': 'abl-pavia-x4-l1', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --w_sam 0 --w_ssim 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect'}, {'name': 'abl-cave-x4-learned', 'dataset': 'cave', 'scale': 4, 'model': 'puformer', 'args': '--dataset cave --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --op learned', 'hours': 4.0, 'robust': True, 'rargs': '--dataset cave --scale 4 --model puformer --pad reflect --op learned'}], [{'name': 'bench-cave-x4-psrt', 'dataset': 'cave', 'scale': 4, 'model': 'psrt', 'args': '--dataset cave --scale 4 --model psrt --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset cave --scale 4 --model psrt'}, {'name': 'bench-cave-x4-mimformer', 'dataset': 'cave', 'scale': 4, 'model': 'mimformer', 'args': '--dataset cave --scale 4 --model mimformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset cave --scale 4 --model mimformer'}, {'name': 'bench-harvard-x4-psrt', 'dataset': 'harvard', 'scale': 4, 'model': 'psrt', 'args': '--dataset harvard --scale 4 --model psrt --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset harvard --scale 4 --model psrt'}, {'name': 'bench-harvard-x4-mimformer', 'dataset': 'harvard', 'scale': 4, 'model': 'mimformer', 'args': '--dataset harvard --scale 4 --model mimformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset harvard --scale 4 --model mimformer'}, {'name': 'bench-pavia-x8-psrt', 'dataset': 'pavia', 'scale': 8, 'model': 'psrt', 'args': '--dataset pavia --scale 8 --model psrt --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 8 --model psrt'}, {'name': 'bench-pavia-x8-mimformer', 'dataset': 'pavia', 'scale': 8, 'model': 'mimformer', 'args': '--dataset pavia --scale 8 --model mimformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 8 --model mimformer'}, {'name': 'bench-chikusei-x8-puformer', 'dataset': 'chikusei', 'scale': 8, 'model': 'puformer', 'args': '--dataset chikusei --scale 8 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1', 'hours': 4.0, 'robust': True, 'rargs': '--dataset chikusei --scale 8 --model puformer --pad reflect'}, {'name': 'bench-harvard-x8-mimformer', 'dataset': 'harvard', 'scale': 8, 'model': 'mimformer', 'args': '--dataset harvard --scale 8 --model mimformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset harvard --scale 8 --model mimformer'}, {'name': 'bench-pavia-x16-psrt', 'dataset': 'pavia', 'scale': 16, 'model': 'psrt', 'args': '--dataset pavia --scale 16 --model psrt --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 16 --model psrt'}, {'name': 'bench-pavia-x16-mimformer', 'dataset': 'pavia', 'scale': 16, 'model': 'mimformer', 'args': '--dataset pavia --scale 16 --model mimformer --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 16 --model mimformer'}, {'name': 'bench-chikusei-x16-psrt', 'dataset': 'chikusei', 'scale': 16, 'model': 'psrt', 'args': '--dataset chikusei --scale 16 --model psrt --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset chikusei --scale 16 --model psrt'}, {'name': 'bench-chikusei-x16-mimformer', 'dataset': 'chikusei', 'scale': 16, 'model': 'mimformer', 'args': '--dataset chikusei --scale 16 --model mimformer --patch 128 --bs 4 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset chikusei --scale 16 --model mimformer'}, {'name': 'abl-pavia-x4-nores', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --res_input 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --res_input 0'}, {'name': 'abl-pavia-x4-learned', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --op learned', 'hours': 4.0, 'robust': True, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --op learned'}, {'name': 'abl-pavia-x4-k1', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --stages 1', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --stages 1'}, {'name': 'abl-pavia-x4-k4', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --stages 4', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --stages 4'}, {'name': 'abl-pavia-x4-window', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --mixer window', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect --mixer window'}, {'name': 'abl-pavia-x4-nodeep', 'dataset': 'pavia', 'scale': 4, 'model': 'puformer', 'args': '--dataset pavia --scale 4 --model puformer --bs 8 --seed 0 --eval_every 1000 --log_every 1000 --test_log 0 --pad reflect --w_sam 0.05 --w_ssim 0.1 --w_deep 0', 'hours': 4.0, 'robust': False, 'rargs': '--dataset pavia --scale 4 --model puformer --pad reflect'}]]
SMOKE = True
SMOKE_ROBUST = ('bench-cave-x4-puformer', 'abl-pavia-x4-learned', 'bench-pavia-x8-puformer')


def find(pattern):
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True), key=len)
    return hits[0] if hits else None


chik = [p for p in glob.glob('/kaggle/input/**/*.mat', recursive=True) if 'chikusei' in p.lower() and 'Ground' not in p]
DATA = dict(chikusei=sorted(chik, key=len)[0] if chik else None, pavia=find('Pavia.mat'),
            cave=os.path.dirname(os.path.dirname(os.path.dirname(find('Train/HSI/balloons.mat') or '/x/x/x/x'))),
            harvard=os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(find('Data/Train/HSI/img*.mat') or '/x/x/x/x/x')))))
CACHE = dict(chikusei='/tmp/chikusei.npy', harvard='/tmp/harvard')
print('data:', DATA, flush=True)
ENV = dict(os.environ, OMP_NUM_THREADS='2', PUF_EXT='/kaggle/working/ext', PYTHONUNBUFFERED='1')


In [ ]:
# 1) self-checks, data caches shared by both GPUs, the published networks' code
assert subprocess.run('python selfcheck.py', shell=True, cwd=CODE, env=ENV).returncode == 0
used = {j['dataset'] for q in QUEUES for j in q}
for d in used:
    if d in CACHE:   # one process builds the cache, the GPU jobs memory-map it
        t = time.time()
        rc = subprocess.run([sys.executable, '-c', f"from hsifuse.data import load_dataset; load_dataset({d!r}, {DATA[d]!r}, {CACHE[d]!r})"],
                            cwd=CODE, env=ENV).returncode
        print(d, 'cache built, rc', rc, f'{time.time() - t:.0f} s', flush=True)
models = {j['model'] for q in QUEUES for j in q} - {'puformer', 'ssrnet'}
sys.path.insert(0, CODE)
os.environ['PUF_EXT'] = ENV['PUF_EXT']
from hsifuse.external import fetch
for m in sorted(models):
    fetch(m)


In [ ]:
# 2) the two GPU queues
lock = threading.Lock()


def say(*s):
    with lock:
        print(*s, flush=True)


def sh(cmd, gpu, log):
    env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))
    with open(log, 'a') as f:
        p = subprocess.Popen(cmd, shell=True, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if any(k in line for k in ('ep ', 'TEST', 'consistency', 'Error', 'error', 'Traceback', 'PROBE', 'nominal',
                                       'sigma', 'noise', 'srf', 'warm', 'params on', 'train ', 'test:', 'budget', 'it/s', 'external')):
                say(f'[gpu{gpu}]', line.rstrip()[:400])
        return p.wait()


def worker(gpu, queue):
    for i, j in enumerate(queue):
        out = f"{OUT}/{j['name']}"
        os.makedirs(out, exist_ok=True)
        log = f'{out}/train.log'
        left = LIMIT - time.time()
        reserve = 1800 * (len(queue) - i)                    # final test + robustness of this and later jobs
        budget = min(j['hours'] * 3600, (left - reserve) / (len(queue) - i) if i < len(queue) - 1 else left - 1800)
        if budget < 1200 and not SMOKE:
            say(f'[gpu{gpu}] skip {j["name"]}: only {budget:.0f} s left'); continue
        deadline = time.time() + budget
        cache = f"--cache {CACHE[j['dataset']]}" if j['dataset'] in CACHE else ''
        data = f"--mat {shlex.quote(DATA[j['dataset']])} {cache}"
        stop = '--iters 60 --eval_every 30 --log_every 20 --q2n 0' if SMOKE else f'--deadline {deadline}'
        say(f'[gpu{gpu}] START {j["name"]} budget {budget / 3600:.2f} h: {j["args"]}')
        rc = sh(f"python train.py {data} {j['args']} {stop} --out {out}", gpu, log)
        say(f'[gpu{gpu}] END {j["name"]} rc={rc} after {(time.time() - T0) / 3600:.2f} h')
        if rc == 0 and j['robust'] and time.time() < LIMIT - 900 and (not SMOKE or j['name'] in SMOKE_ROBUST):
            sh(f"python eval_gaps.py {data} {j['rargs']} --ckpt {out}/best_ema.pt --out {out}", gpu, log)
        for f in ['last.pt', 'ema_last.pt', 'test_pred_tta.npy'] + [os.path.basename(p) for p in glob.glob(f'{out}/ema_ep*.pt')]:
            if os.path.exists(f'{out}/{f}'):
                os.remove(f'{out}/{f}')
        pred = f'{out}/test_pred.npy'
        if os.path.exists(pred):
            import numpy as np
            np.save(f'{out}/test_pred_f16.npy', np.load(pred).astype(np.float16)); os.remove(pred)


threads = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(QUEUES)]
for t in threads:
    t.start()
for t in threads:
    t.join()


In [ ]:
# 3) summary
for q in QUEUES:
    for j in q:
        f = f"{OUT}/{j['name']}/results.json"
        if not os.path.exists(f):
            print(j['name'], 'no results'); continue
        r = json.load(open(f))
        t, tt = r['test'], r['test_tta']
        print(f"{j['name']:34s} it {r['iters']:6d} {r['params_M']:.2f}M val {r['best_val_PSNR']:.3f} "
              f"| PSNR {t['PSNR']:.3f}/{tt['PSNR']:.3f} SAM {t['SAM']:.3f} ERGAS {t['ERGAS']:.3f} "
              f"SSIM {t['SSIM']:.4f} RMSE {t['RMSE_DN']:.2f} | cons {r['consistency']}")
subprocess.run(f'du -sh {OUT}; rm -rf /kaggle/working/ext', shell=True)
